## Gemini

Push VQA to Google Gemini.

### Docs:

* in theory, there is documentation here: https://ai.google.dev/gemini-api/docs, 
* Here though, I just asked claude again :D

#### Key Points (cp-claude)

* API Key: Get your API key from the [Google Console](https://aistudio.google.com/app/apikey) -- note that you either have to create a new GCP project OR create the API key in an existing project
* Then the rest of the steps is similar for Claude/ChatGPT


First, install anthropic api (also, see .yml file for the environment for this project)

In [1]:
#!pip install google-genai

Where are things stored/going to be stored?

What is thinking budget:

| Model | Min | Max | Can disable (=0)? |
|---|---|---|---|
| `gemini-2.5-flash` | 1 | 24,576 | ✅ yes |
| `gemini-2.5-flash-lite` | 512 | 24,576 | ✅ yes |
| `gemini-2.5-pro` | 128 | 32,768 | ❌ no |

Setting `thinking_budget` to `-1` turns on dynamic thinking, where the model automatically adjusts the budget based on the complexity of the request. Google AI When dynamic thinking is enabled this way, the default max is 8,192 tokens.

In [2]:
# ============================ RUN CONFIG ============================
# LOW-TIER run.  This is the paper's `gemini/` baseline -- Gemini only had the
# one configuration, so it served as the low tier.
#
# MODEL CHANGE (required, not optional): the previous model
# "gemini-3.1-flash-lite-preview" was SHUT DOWN on 2026-05-25 and now errors.
# Google's stated replacement is "gemini-3.1-flash-lite"; the newer low-tier
# flash-lite is "gemini-3.5-flash-lite", which is what we move to here.
out_base = '~/astro_sky_image_vqa/LMM_outputs_n255/'   # TEST tree, not the real one

dir_api = out_base + 'gemini/'
model_name = 'gemini-3.5-flash-lite'   # was gemini-3.1-flash-lite-preview (retired)

# PARAMETER CHANGE: Gemini 3.x uses `thinking_level`, not the 2.5-era
# `thinking_budget`.  The two are mutually exclusive -- sending both returns a
# 400.  The old notebook was still passing thinking_budget (a legacy parameter)
# to a Gemini 3 model, so this is a required update, not a preference.
#   thinking_level: "minimal" | "low" | "medium" | "high"
#
# "minimal" for the low tier, so all three runs are aligned on no reasoning:
#   chatgpt low -> reasoning_level = None     (no reasoning effort sent)
#   claude  low -> thinking = False           (no thinking block)
#   gemini  low -> thinking_level = 'minimal' (least thinking available)
#
# NOTE: this differs from the ORIGINAL low-tier gemini run, which used
# thinking_budget = -1 (dynamic thinking) and so was strictly more capable than
# the other two low-tier runs.  Gemini numbers are therefore not directly
# comparable with the previous paper's gemini column.
thinking_level = 'minimal'

# ---- reference ----
# model_name = 'gemini-3.1-flash-lite'   # Google's direct replacement for the retired preview
# thinking_level = 'high'                # closest to the ORIGINAL run's dynamic thinking
#
# NOTE: temperature / top_p / top_k were deprecated across Gemini 3.x on
# 2026-07-21.  This notebook does not set them -- do not reintroduce them.

key_file = '/Users/jnaiman/.gemini/key.txt'

# where is VQA dataset?
jsons_dir = '~/astro_sky_image_vqa/VQA_full/qa_jsons/' # directory where jsons created with figure are stored
imgs_dir = '~/astro_sky_image_vqa/VQA_full/imgs/' # where images are stored

# for saving temp images for reading in
tmp_dir = '/Users/jnaiman/Downloads/tmp/' # this might not be used...

img_format = 'jpeg'

# for asking for reasoning
reasoning_text = 'In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.'


In [3]:
import base64
from PIL import Image
import numpy as np
import json
import re
import pickle
import os
from glob import glob
# import google.generativeai as genai
from google import genai
from google.genai import types
# typed errors, so a 429 can be told apart from a 400 (see send_to_gemini)
from google.genai.errors import ClientError, ServerError

# debug
from importlib import reload
from copy import deepcopy


from sys import path
path.append('../')
import utils.llm_utils
reload(utils.llm_utils)
from utils.llm_utils import parse_qa, load_image, get_img_json_pair, parse_for_errors

import time
from utils.plot_qa_utils import get_nplots

# --- expand ~ ONCE, and assign back ----------------------------------------
# These must be rebound, not expanded locally: everything downstream (glob,
# open, os.path.join) uses the variables directly, and glob('~/...') silently
# matches nothing rather than erroring.
out_base  = os.path.expanduser(out_base)
dir_api   = os.path.expanduser(dir_api)
key_file  = os.path.expanduser(key_file)
jsons_dir = os.path.expanduser(jsons_dir)
imgs_dir  = os.path.expanduser(imgs_dir)
tmp_dir   = os.path.expanduser(tmp_dir)

# --- make the directories we WRITE to -------------------------------------
# dir_api holds one <id>_qa.pickle per figure; tmp_dir holds the resized images
# load_image() falls back to when a request is too large.  Neither is created
# by anything upstream, so a fresh out_base fails on the first save.
for _d in (dir_api, tmp_dir):
    if not os.path.exists(_d):
        os.makedirs(_d, exist_ok=True)   # makedirs, not mkdir: the path is nested
        print('made:', _d)

# --- check the directories we READ from ------------------------------------
for _name, _d in (('jsons_dir', jsons_dir), ('imgs_dir', imgs_dir)):
    if not os.path.isdir(_d):
        print('[WARN] %s does not exist: %s' % (_name, _d))
if not os.path.exists(key_file):
    print('[WARN] key_file does not exist:', key_file)


In [4]:
# setup
with open(key_file,'r') as f:
    api_key = f.read()

# Configure the API key
client = genai.Client(api_key=api_key.strip())

# NOTE: previously a GenerateContentConfig was built here with ONLY
# thinking_config and passed into send_to_gemini().  Because send_to_gemini only
# fills in system_instruction when config is None, passing this meant the
# persona/system prompt was silently never sent to Gemini -- while ChatGPT and
# Claude both received it.  Leave this as None so the per-question config built
# inside send_to_gemini (which does set system_instruction) is used.
config = None

# thinking level is applied per-call inside send_to_gemini


In [5]:
#model_gemini = genai.GenerativeModel(model)

In [6]:
# ================= QUICK TEST: five figures per category =================
# The released VQA ids are shuffled and carry no category (deliberately -- the
# generator's own names encoded the real-vs-synthetic answer). So classify by
# reading each qa json instead of by filename.
#
# TEST_IDS pins the exact figures so the test is reproducible, and is the SAME
# set used by the other test notebooks, so the models are comparable. Each
# value may be a single id or a list of them. Set TEST_IDS = None to auto-pick
# the first N_PER_CATEGORY of each category and print a fresh block.

# ---- the original one-per-category run; uncomment this block (and comment out
# ---- the five-per-category one below) to go back to a 3-figure test ----
# TEST_IDS = {
#     'contour':  'vqa_000003',
#     'sky-real': 'vqa_000001',
#     'sky-gmm':  'vqa_000004',
# }

# ---- the five-per-category run; uncomment to go back to a 15-figure test ----
# TEST_IDS = {
#     'contour':  ['vqa_000003', 'vqa_000006', 'vqa_000010', 'vqa_000012', 'vqa_000014'],
#     'sky-real': ['vqa_000001', 'vqa_000002', 'vqa_000009', 'vqa_000021', 'vqa_000022'],
#     'sky-gmm':  ['vqa_000004', 'vqa_000005', 'vqa_000007', 'vqa_000008', 'vqa_000011'],
# }

# Fifty per category = 150 figures.  These are DISJOINT from every id used by the
# 3- and 15-figure runs above, so no figure is scored twice across the test sets;
# they are the first 50 of each category, in sorted id order, skipping those.
# ---- the previous fifty-per-category run.  Those ids came from the SUPERSEDED
# ---- lookup table (archived as vqa_lookup_superseded_*.json): the dataset was
# ---- regenerated afterwards and every id was reassigned, so they no longer
# ---- point at the figures they were scored on.  Kept for reference only.
# TEST_IDS = {
#     'contour':  ['vqa_000016', 'vqa_000017', 'vqa_000018', 'vqa_000019', 'vqa_000020',
#                  'vqa_000025', 'vqa_000029', 'vqa_000032', 'vqa_000033', 'vqa_000035',
#                  'vqa_000038', 'vqa_000039', 'vqa_000041', 'vqa_000043', 'vqa_000045',
#                  'vqa_000049', 'vqa_000050', 'vqa_000053', 'vqa_000055', 'vqa_000056',
#                  'vqa_000064', 'vqa_000066', 'vqa_000068', 'vqa_000069', 'vqa_000070',
#                  'vqa_000072', 'vqa_000084', 'vqa_000085', 'vqa_000088', 'vqa_000091',
#                  'vqa_000097', 'vqa_000103', 'vqa_000105', 'vqa_000106', 'vqa_000111',
#                  'vqa_000112', 'vqa_000113', 'vqa_000118', 'vqa_000121', 'vqa_000124',
#                  'vqa_000135', 'vqa_000141', 'vqa_000143', 'vqa_000144', 'vqa_000146',
#                  'vqa_000151', 'vqa_000154', 'vqa_000155', 'vqa_000156', 'vqa_000158'],
#     'sky-real': ['vqa_000028', 'vqa_000036', 'vqa_000037', 'vqa_000040', 'vqa_000042',
#                  'vqa_000044', 'vqa_000047', 'vqa_000048', 'vqa_000052', 'vqa_000054',
#                  'vqa_000057', 'vqa_000059', 'vqa_000060', 'vqa_000063', 'vqa_000074',
#                  'vqa_000076', 'vqa_000078', 'vqa_000079', 'vqa_000082', 'vqa_000083',
#                  'vqa_000087', 'vqa_000092', 'vqa_000093', 'vqa_000095', 'vqa_000096',
#                  'vqa_000098', 'vqa_000101', 'vqa_000102', 'vqa_000107', 'vqa_000109',
#                  'vqa_000114', 'vqa_000115', 'vqa_000116', 'vqa_000117', 'vqa_000120',
#                  'vqa_000125', 'vqa_000126', 'vqa_000128', 'vqa_000130', 'vqa_000131',
#                  'vqa_000132', 'vqa_000133', 'vqa_000134', 'vqa_000137', 'vqa_000138',
#                  'vqa_000140', 'vqa_000149', 'vqa_000150', 'vqa_000152', 'vqa_000157'],
#     'sky-gmm':  ['vqa_000013', 'vqa_000015', 'vqa_000023', 'vqa_000024', 'vqa_000026',
#                  'vqa_000027', 'vqa_000030', 'vqa_000031', 'vqa_000034', 'vqa_000046',
#                  'vqa_000051', 'vqa_000058', 'vqa_000061', 'vqa_000062', 'vqa_000065',
#                  'vqa_000067', 'vqa_000071', 'vqa_000073', 'vqa_000075', 'vqa_000077',
#                  'vqa_000080', 'vqa_000081', 'vqa_000086', 'vqa_000089', 'vqa_000090',
#                  'vqa_000094', 'vqa_000099', 'vqa_000100', 'vqa_000104', 'vqa_000108',
#                  'vqa_000110', 'vqa_000119', 'vqa_000122', 'vqa_000123', 'vqa_000127',
#                  'vqa_000129', 'vqa_000136', 'vqa_000139', 'vqa_000142', 'vqa_000145',
#                  'vqa_000147', 'vqa_000148', 'vqa_000153', 'vqa_000159', 'vqa_000163',
#                  'vqa_000164', 'vqa_000166', 'vqa_000167', 'vqa_000169', 'vqa_000170'],
# }

# Fifty per category = 150 figures, drawn from the REBUILT lookup table (the one
# covering the regenerated dataset: TESS excluded, real fields given a deliberate
# angular size, gmm field sizes and colour scales matched to the real family).
# Taken as the first 50 of each category in sorted id order -- the ids are
# already a single-pass shuffle of the source figures, so that is an unbiased
# sample and needs no separate seed.  The SAME 150 are used by every test
# notebook, so the three models and the three aging levels stay comparable.
# ---- the superseded FIFTY-per-category selection (150 figures).
# ---- Every one of these ids is still in TEST_IDS below; this block is
# ---- kept only to show what the earlier runs covered.
# TEST_IDS = {
#     'contour':  ['vqa_000007', 'vqa_000010', 'vqa_000011', 'vqa_000013', 'vqa_000017',
#                  'vqa_000023', 'vqa_000024', 'vqa_000025', 'vqa_000030', 'vqa_000032',
#                  'vqa_000034', 'vqa_000035', 'vqa_000038', 'vqa_000040', 'vqa_000041',
#                  'vqa_000042', 'vqa_000052', 'vqa_000057', 'vqa_000061', 'vqa_000063',
#                  'vqa_000065', 'vqa_000067', 'vqa_000070', 'vqa_000072', 'vqa_000077',
#                  'vqa_000078', 'vqa_000087', 'vqa_000088', 'vqa_000089', 'vqa_000090',
#                  'vqa_000092', 'vqa_000094', 'vqa_000095', 'vqa_000097', 'vqa_000101',
#                  'vqa_000102', 'vqa_000103', 'vqa_000105', 'vqa_000107', 'vqa_000109',
#                  'vqa_000110', 'vqa_000111', 'vqa_000115', 'vqa_000127', 'vqa_000128',
#                  'vqa_000132', 'vqa_000136', 'vqa_000140', 'vqa_000141', 'vqa_000146'],
#     'sky-real': ['vqa_000002', 'vqa_000003', 'vqa_000004', 'vqa_000005', 'vqa_000006',
#                  'vqa_000008', 'vqa_000009', 'vqa_000015', 'vqa_000018', 'vqa_000019',
#                  'vqa_000020', 'vqa_000027', 'vqa_000028', 'vqa_000029', 'vqa_000031',
#                  'vqa_000033', 'vqa_000036', 'vqa_000044', 'vqa_000045', 'vqa_000046',
#                  'vqa_000049', 'vqa_000050', 'vqa_000055', 'vqa_000066', 'vqa_000074',
#                  'vqa_000080', 'vqa_000082', 'vqa_000086', 'vqa_000091', 'vqa_000093',
#                  'vqa_000096', 'vqa_000098', 'vqa_000099', 'vqa_000100', 'vqa_000104',
#                  'vqa_000106', 'vqa_000108', 'vqa_000117', 'vqa_000118', 'vqa_000120',
#                  'vqa_000121', 'vqa_000122', 'vqa_000124', 'vqa_000125', 'vqa_000126',
#                  'vqa_000129', 'vqa_000133', 'vqa_000137', 'vqa_000138', 'vqa_000139'],
#     'sky-gmm':  ['vqa_000001', 'vqa_000012', 'vqa_000014', 'vqa_000016', 'vqa_000021',
#                  'vqa_000022', 'vqa_000026', 'vqa_000037', 'vqa_000039', 'vqa_000043',
#                  'vqa_000047', 'vqa_000048', 'vqa_000051', 'vqa_000053', 'vqa_000054',
#                  'vqa_000056', 'vqa_000058', 'vqa_000059', 'vqa_000060', 'vqa_000062',
#                  'vqa_000064', 'vqa_000068', 'vqa_000069', 'vqa_000071', 'vqa_000073',
#                  'vqa_000075', 'vqa_000076', 'vqa_000079', 'vqa_000081', 'vqa_000083',
#                  'vqa_000084', 'vqa_000085', 'vqa_000112', 'vqa_000113', 'vqa_000114',
#                  'vqa_000116', 'vqa_000119', 'vqa_000123', 'vqa_000130', 'vqa_000131',
#                  'vqa_000134', 'vqa_000135', 'vqa_000142', 'vqa_000143', 'vqa_000144',
#                  'vqa_000145', 'vqa_000149', 'vqa_000151', 'vqa_000156', 'vqa_000157'],
# }

# Eighty-five per category = 255 figures.  Grown from the previous 150 by
# keeping ALL 150 of the earlier ids untouched and drawing 35 MORE per category
# at random from the rest of the dataset (seed 20261008, chosen over the 617
# remaining eligible figures in each category).
#
# Keeping the old ids verbatim is what makes this cheap to run: their pickles
# already exist, and the run cell below has restart=False, so only the 105 new
# figures are actually sent to the API.  Re-drawing all 255 would have thrown
# away 150 figures' worth of paid-for answers.
#
# The SAME 255 are used by every test notebook, so the three models and the two
# aging levels stay comparable.  Verified before the draw: all 150 original ids
# are still present in the regenerated dataset and still fall in the category
# they were originally scored under.
TEST_IDS = {
    'contour': ['vqa_000007', 'vqa_000010', 'vqa_000011', 'vqa_000013', 'vqa_000017',
                'vqa_000023', 'vqa_000024', 'vqa_000025', 'vqa_000030', 'vqa_000032',
                'vqa_000034', 'vqa_000035', 'vqa_000038', 'vqa_000040', 'vqa_000041',
                'vqa_000042', 'vqa_000052', 'vqa_000057', 'vqa_000061', 'vqa_000063',
                'vqa_000065', 'vqa_000067', 'vqa_000070', 'vqa_000072', 'vqa_000077',
                'vqa_000078', 'vqa_000087', 'vqa_000088', 'vqa_000089', 'vqa_000090',
                'vqa_000092', 'vqa_000094', 'vqa_000095', 'vqa_000097', 'vqa_000101',
                'vqa_000102', 'vqa_000103', 'vqa_000105', 'vqa_000107', 'vqa_000109',
                'vqa_000110', 'vqa_000111', 'vqa_000115', 'vqa_000127', 'vqa_000128',
                'vqa_000132', 'vqa_000136', 'vqa_000140', 'vqa_000141', 'vqa_000146',
                'vqa_000180', 'vqa_000255', 'vqa_000296', 'vqa_000333', 'vqa_000337',
                'vqa_000391', 'vqa_000417', 'vqa_000441', 'vqa_000537', 'vqa_000609',
                'vqa_000732', 'vqa_000790', 'vqa_000805', 'vqa_000817', 'vqa_000850',
                'vqa_000860', 'vqa_000975', 'vqa_001007', 'vqa_001021', 'vqa_001027',
                'vqa_001088', 'vqa_001231', 'vqa_001261', 'vqa_001371', 'vqa_001391',
                'vqa_001402', 'vqa_001494', 'vqa_001519', 'vqa_001549', 'vqa_001550',
                'vqa_001556', 'vqa_001627', 'vqa_001635', 'vqa_001836', 'vqa_001864'],
    'sky-real': ['vqa_000002', 'vqa_000003', 'vqa_000004', 'vqa_000005', 'vqa_000006',
                 'vqa_000008', 'vqa_000009', 'vqa_000015', 'vqa_000018', 'vqa_000019',
                 'vqa_000020', 'vqa_000027', 'vqa_000028', 'vqa_000029', 'vqa_000031',
                 'vqa_000033', 'vqa_000036', 'vqa_000044', 'vqa_000045', 'vqa_000046',
                 'vqa_000049', 'vqa_000050', 'vqa_000055', 'vqa_000066', 'vqa_000074',
                 'vqa_000080', 'vqa_000082', 'vqa_000086', 'vqa_000091', 'vqa_000093',
                 'vqa_000096', 'vqa_000098', 'vqa_000099', 'vqa_000100', 'vqa_000104',
                 'vqa_000106', 'vqa_000108', 'vqa_000117', 'vqa_000118', 'vqa_000120',
                 'vqa_000121', 'vqa_000122', 'vqa_000124', 'vqa_000125', 'vqa_000126',
                 'vqa_000129', 'vqa_000133', 'vqa_000137', 'vqa_000138', 'vqa_000139',
                 'vqa_000155', 'vqa_000187', 'vqa_000212', 'vqa_000261', 'vqa_000299',
                 'vqa_000322', 'vqa_000327', 'vqa_000347', 'vqa_000387', 'vqa_000392',
                 'vqa_000393', 'vqa_000434', 'vqa_000450', 'vqa_000486', 'vqa_000508',
                 'vqa_000538', 'vqa_000654', 'vqa_000701', 'vqa_000736', 'vqa_000772',
                 'vqa_000941', 'vqa_001015', 'vqa_001064', 'vqa_001138', 'vqa_001145',
                 'vqa_001163', 'vqa_001215', 'vqa_001487', 'vqa_001493', 'vqa_001500',
                 'vqa_001514', 'vqa_001771', 'vqa_001797', 'vqa_001798', 'vqa_001937'],
    'sky-gmm': ['vqa_000001', 'vqa_000012', 'vqa_000014', 'vqa_000016', 'vqa_000021',
                'vqa_000022', 'vqa_000026', 'vqa_000037', 'vqa_000039', 'vqa_000043',
                'vqa_000047', 'vqa_000048', 'vqa_000051', 'vqa_000053', 'vqa_000054',
                'vqa_000056', 'vqa_000058', 'vqa_000059', 'vqa_000060', 'vqa_000062',
                'vqa_000064', 'vqa_000068', 'vqa_000069', 'vqa_000071', 'vqa_000073',
                'vqa_000075', 'vqa_000076', 'vqa_000079', 'vqa_000081', 'vqa_000083',
                'vqa_000084', 'vqa_000085', 'vqa_000112', 'vqa_000113', 'vqa_000114',
                'vqa_000116', 'vqa_000119', 'vqa_000123', 'vqa_000130', 'vqa_000131',
                'vqa_000134', 'vqa_000135', 'vqa_000142', 'vqa_000143', 'vqa_000144',
                'vqa_000145', 'vqa_000149', 'vqa_000151', 'vqa_000156', 'vqa_000157',
                'vqa_000178', 'vqa_000202', 'vqa_000281', 'vqa_000356', 'vqa_000358',
                'vqa_000389', 'vqa_000477', 'vqa_000546', 'vqa_000560', 'vqa_000600',
                'vqa_000747', 'vqa_000884', 'vqa_000904', 'vqa_000920', 'vqa_000960',
                'vqa_000973', 'vqa_000977', 'vqa_001013', 'vqa_001035', 'vqa_001039',
                'vqa_001072', 'vqa_001078', 'vqa_001096', 'vqa_001183', 'vqa_001254',
                'vqa_001266', 'vqa_001594', 'vqa_001602', 'vqa_001743', 'vqa_001746',
                'vqa_001785', 'vqa_001856', 'vqa_001916', 'vqa_001986', 'vqa_002000'],
}
# TEST_IDS = None   # auto-pick instead

CATEGORIES = ['contour', 'sky-real', 'sky-gmm']
N_PER_CATEGORY = 85       # only consulted when TEST_IDS is None


def categorise_qa_json(qa_json_path):
    """contour / sky-real / sky-gmm, from the panel types + distributions."""
    with open(qa_json_path, 'r') as f:
        d = json.loads(json.load(f))
    types, dists = set(), set()
    for k, v in d.items():
        if k.startswith('plot'):
            types.add(v.get('type'))
            dists.add(v.get('distribution'))
    if types == {'contour'}:
        return 'contour'
    if types == {'image of the sky'}:
        if dists == {'sky'}:
            return 'sky-real'
        if dists == {'gmm'}:
            return 'sky-gmm'
        return 'sky-mixed'
    return 'other'


def has_image(qa_json_path):
    stem = os.path.basename(qa_json_path).removesuffix('_qa.json')
    return os.path.exists(os.path.join(imgs_dir, stem + '.' + img_format))


all_qa = sorted(glob(os.path.join(jsons_dir, '*.json')))
all_qa = [j for j in all_qa if has_image(j)]
print('total qa files with a matching image:', len(all_qa))

jsons_to_parse = []
if TEST_IDS:
    for cat in CATEGORIES:
        vids = TEST_IDS.get(cat) or []
        if isinstance(vids, str):        # a bare id, as the 3-figure block uses
            vids = [vids]
        for vid in vids:
            pth = os.path.join(jsons_dir, vid + '_qa.json')
            if not os.path.exists(pth):
                print('[WARN] %s (%s) not found -- skipping' % (vid, cat))
                continue
            jsons_to_parse.append(pth)
else:
    # first N_PER_CATEGORY of each category, in sorted order -- deterministic
    picked = {c: [] for c in CATEGORIES}
    for j in all_qa:
        c = categorise_qa_json(j)
        if c in picked and len(picked[c]) < N_PER_CATEGORY:
            picked[c].append(j)
        if all(len(v) >= N_PER_CATEGORY for v in picked.values()):
            break
    jsons_to_parse = [j for c in CATEGORIES for j in picked[c]]
    print()
    print('# paste into TEST_IDS above to pin this selection:')
    print('TEST_IDS = {')
    for c in CATEGORIES:
        ids = [os.path.basename(j).removesuffix('_qa.json') for j in picked[c]]
        print("    %-11s %r," % ("'%s':" % c, ids))
    print('}')

print()
print('testing on %d figures:' % len(jsons_to_parse))
for j in jsons_to_parse:
    print('   %-10s %s' % (categorise_qa_json(j), os.path.basename(j)))


total qa files with a matching image: 2001

testing on 255 figures:
   contour    vqa_000007_qa.json
   contour    vqa_000010_qa.json
   contour    vqa_000011_qa.json
   contour    vqa_000013_qa.json
   contour    vqa_000017_qa.json
   contour    vqa_000023_qa.json
   contour    vqa_000024_qa.json


   contour    vqa_000025_qa.json
   contour    vqa_000030_qa.json
   contour    vqa_000032_qa.json
   contour    vqa_000034_qa.json
   contour    vqa_000035_qa.json
   contour    vqa_000038_qa.json
   contour    vqa_000040_qa.json
   contour    vqa_000041_qa.json
   contour    vqa_000042_qa.json


   contour    vqa_000052_qa.json
   contour    vqa_000057_qa.json
   contour    vqa_000061_qa.json
   contour    vqa_000063_qa.json
   contour    vqa_000065_qa.json
   contour    vqa_000067_qa.json
   contour    vqa_000070_qa.json
   contour    vqa_000072_qa.json


   contour    vqa_000077_qa.json
   contour    vqa_000078_qa.json
   contour    vqa_000087_qa.json
   contour    vqa_000088_qa.json
   contour    vqa_000089_qa.json
   contour    vqa_000090_qa.json
   contour    vqa_000092_qa.json
   contour    vqa_000094_qa.json


   contour    vqa_000095_qa.json
   contour    vqa_000097_qa.json
   contour    vqa_000101_qa.json
   contour    vqa_000102_qa.json
   contour    vqa_000103_qa.json
   contour    vqa_000105_qa.json
   contour    vqa_000107_qa.json
   contour    vqa_000109_qa.json


...

   sky-gmm    vqa_000884_qa.json
   sky-gmm    vqa_000904_qa.json
   sky-gmm    vqa_000920_qa.json
   sky-gmm    vqa_000960_qa.json
   sky-gmm    vqa_000973_qa.json


   sky-gmm    vqa_000977_qa.json
   sky-gmm    vqa_001013_qa.json
   sky-gmm    vqa_001035_qa.json
   sky-gmm    vqa_001039_qa.json
   sky-gmm    vqa_001072_qa.json
   sky-gmm    vqa_001078_qa.json


   sky-gmm    vqa_001096_qa.json
   sky-gmm    vqa_001183_qa.json
   sky-gmm    vqa_001254_qa.json
   sky-gmm    vqa_001266_qa.json
   sky-gmm    vqa_001594_qa.json
   sky-gmm    vqa_001602_qa.json


   sky-gmm    vqa_001743_qa.json
   sky-gmm    vqa_001746_qa.json
   sky-gmm    vqa_001785_qa.json
   sky-gmm    vqa_001856_qa.json


   sky-gmm    vqa_001916_qa.json
   sky-gmm    vqa_001986_qa.json
   sky-gmm    vqa_002000_qa.json


In [7]:
def send_to_gemini(question_list, image_path, client,model_name='gemini-3.5-flash-lite',
                    test_run = True, 
                    verbose=True, verbose_tokens = False,
                    system_prompt = None, img_format='jpg',         
                   large_image= False, config=None, reasoning=None,
                   thinking_level=None,
                   max_retries=10, sleep_time=1):
    """
    Sends an image + question to Gemini.  Does something different for large file, but might be bad idea.

    system_prompt : set to None to use default from question list
    """
    if system_prompt is None:
        system_prompt = question_list['persona']
    if img_format.lower() == 'jpg':
        img_format = 'jpeg'

    err = False
    #model_gemini = genai.GenerativeModel(model)
    #if verbose: print('Model loaded:', model)
    prompt_save = ''; prompt = ''; response = ''
    if not test_run:
        question = question_list['context'] + " " + question_list['question'] + " " + question_list['format']
        if reasoning is not None:
            question += " " + reasoning
        # lowercase the first word, just in case
        question = question.lstrip() # no whitespace
        question = question[0].lower() + question[1:]

        if verbose: print('   on question:',question)
        # Prepare the API request
        prompt = f"I am going to show you an image. Now, {question}"
        prompt_save = f"I am going to show you an image. Now, {question}"
        # --- retry with jittered exponential backoff -----------------------
        # This used to be a bare try/except: any failure, a 429 included, just
        # printed an error and set err, losing that question for this figure.
        # Now rate limits and 5xx are retried the way the claude runner does it.
        # 400s are NOT retried -- waiting does not fix a malformed request.
        attempt = 0
        success = False
        while not success and attempt < max_retries:
            try:
            #if True:

                # # Wait for processing (for video files, you might need to wait longer)
                # if large_image:
                #     # Upload the file to Gemini
                #     #uploaded_file = genai.upload_file(path=image_path)
                #     uploaded_file = client.files.upload(file=image_path)
                #     import time
                #     while uploaded_file.state.name == "PROCESSING":
                #         time.sleep(1)
                #         uploaded_file = genai.get_file(uploaded_file.name)
                # else:
                #     uploaded_file = Image.open(image_path)

                with open(image_path, 'rb') as f:
                    image_bytes = f.read()
                image_part = types.Part.from_bytes(data=image_bytes, mime_type='image/'+img_format)

            
                # Build the config here so system_instruction is ALWAYS applied.
                # (Previously a caller-supplied config skipped this branch entirely
                # and the persona never reached the model.)
                if config is None:
                    cfg_kwargs = {'system_instruction': system_prompt}
                    if thinking_level is not None:
                        # Gemini 3.x: thinking_level, NOT the legacy thinking_budget
                        cfg_kwargs['thinking_config'] = types.ThinkingConfig(
                            thinking_level=thinking_level)
                    config_use = types.GenerateContentConfig(**cfg_kwargs)
                else:
                    # caller supplied one -- make sure it still carries the persona
                    config_use = config
                    if getattr(config_use, 'system_instruction', None) is None:
                        config_use.system_instruction = system_prompt
                # Generate content with the uploaded file
                #response = model_gemini.generate_content([prompt, uploaded_file])
                response = client.models.generate_content(
                    model=model_name,
                    contents=[prompt, image_part],
                    config=config_use
                )
            
                if large_image:
                    # Clean up - delete the uploaded file
                    #genai.delete_file(uploaded_file.name)
                    client.files.delete(name=uploaded_file.name)
            
                #return response.text
        
                success = True

            except ServerError as e:
                # 5xx -- transient on Google's side
                if attempt < max_retries - 1:
                    wait_time = sleep_time*(2 ** attempt) + np.random.uniform(0, 1)
                    print(f"      server error {getattr(e, 'code', '?')}. Waiting {wait_time:.2f}s before retry {attempt + 1}")
                    time.sleep(wait_time)
                    attempt += 1
                else:
                    print('[ERROR]:', str(e))
                    err = True
                    break

            except ClientError as e:
                # 429 is the rate limit; every other 4xx is the caller's fault
                if getattr(e, 'code', None) == 429 and attempt < max_retries - 1:
                    wait_time = sleep_time*(2 ** attempt) + np.random.uniform(0, 1)
                    print(f"      rate limited (429). Waiting {wait_time:.2f}s before retry {attempt + 1}")
                    time.sleep(wait_time)
                    attempt += 1
                else:
                    print('[ERROR]:', str(e))
                    err = True
                    break

            except Exception as e:
                print('[ERROR]:', str(e))
                err = True
                break

            #return f"Error: {str(e)}"
        
    # if not test_run and not err:
    #     for part in response.candidates[0].content.parts:
    #         if not part.thought:  # skip thinking tokens
    #             text = part.text
    if not test_run and not err:
        if response is None:
            print('[ERROR]: response is None')
            err = True
            text = ''
        else:
            text = ''
            for part in response.candidates[0].content.parts:
                if not part.thought:
                    text = part.text
                    break  # take first non-thought part
            if not text:
                print('[WARNING]: no non-thought parts found in response')
        if verbose:
            #print('     response:', str(response.candidates[0].content.parts[0].text).replace('\n',''))
            print('     response:', text)




        question_list['Response (raw)'] = deepcopy(response)
        # Get the response from the API
        #answer = deepcopy(str(response.candidates[0].content.parts[0].text))
        answer = text
        question_list['raw answer'] = answer
        # also calculate usage
        usage = {
            'input_tokens': response.usage_metadata.prompt_token_count,
            'output_tokens': response.usage_metadata.candidates_token_count,
            'total_tokens': response.usage_metadata.total_token_count,
            'cached_content_tokens': getattr(response.usage_metadata, 'cached_content_token_count', 0)
            }

        question_list['usage'] = usage
        if verbose and verbose_tokens:
            print(f"      - Input tokens: {usage.input_tokens}")
            print(f"      - Output tokens: {usage.output_tokens}")
            print(f"      - Total tokens: {usage.input_tokens + usage.output_tokens}")
        # format answer
        if '```json"' in answer:
            answer_format = answer.split('```json"')[-1].split('\n')[0].replace('\n', '')
        elif '```json\n' in answer:
            answer_format = answer.split('```json\n')[-1].split('\n')[0].replace('\n', '')
        elif '```json' in answer:
            answer_format = answer.split('```json')[-1].split('\n')[0].replace('\n', '')
        else:
            answer_format = answer # just give it a shot!
        #answer.replace("```json\n",'').replace("\n```",'')
        try:
            question_list['Response'] = json.loads(answer_format)
        except:
            question_list['Response'] = answer_format
            question_list['Error'] = 'JSON formatting'
        question_list['Response String'] = answer_format
    elif err:
        question_list['raw answer'] = 'ERROR'
    else:
        question_list['Response'] = 'TEST RUN'
        question_list['Response String'] = 'TEST RUN'
        question_list['Response (raw)'] = 'TEST RUN'
    

    return question_list, prompt_save, system_prompt


In [8]:
reload(utils.llm_utils)
from utils.llm_utils import parse_for_errors

iMax = len(jsons_to_parse)   # quick test: just the selected examples
verbose = True
test_run = False # run w/o actually pinging openai
restart = False
# set system_prompt to None to default to what is in question list
system_prompt = """You are a helpful assistant that responds only in valid JSON format. Do not include any explanations, reasoning, or text outside of the JSON response."""
if reasoning_text is not None: # rephrease is reasoning is requested
    system_prompt = """You are a helpful assistant that responds only in valid JSON format. Do not include any text outside of the requested JSON responses."""

#system_prompt = """You must respond with only valid JSON. Start your response immediately with { and end with }. Do not write any text before or after the JSON."""
# temperature=0.1

fac = 1.0

for ijson,json_path in enumerate(jsons_to_parse):
    if ijson >= iMax:
        continue

    print('on', ijson, 'of', min(iMax,len(jsons_to_parse)))

    # get image and base json
    img_path = imgs_dir + json_path.split('/')[-1].removesuffix('_qa.json') + '.' + img_format
    _, img_format_media, base_json, err = get_img_json_pair(img_path, json_path, 
                                                            dir_api, restart=restart,fac=fac,
                                                      tmp_dir=tmp_dir, 
                                                      img_format=img_format) #, load_image=False)
    if err:
        continue
    if verbose: print('Got image!')

    ###### create QA ########
    qa = []
    
    # start with figure level questions
    for k,v in base_json['VQA']['Level 1']['Figure-level questions'].items():
        out = {'Q':v['Q'], 'A':v['A'], 'Level':'Level 1', 'type':'Figure-level questions', 'Response':"", 
               "persona":v['persona'], 'context':v['context'], 'question':v['question'], 'format':v['format'],
               "reasoning":reasoning_text}
        qa.append(out)
    for level in ['Level 2', 'Level 3']:
        if level in base_json['VQA']:
            if 'Figure-level questions' in base_json['VQA'][level]:
                #print('** yes, level ***', level)
                for k,v in base_json['VQA'][level]['Figure-level questions'].items():
                    out = {'Q':v['Q'], 'A':v['A'], 'Level':level, 'type':'Figure-level questions', 'Response':"", 
                        "persona":v['persona'], 'context':v['context'], 'question':v['question'], 'format':v['format'],
                        "reasoning":reasoning_text}
                    qa.append(out)
    
    # what kinds?
    #types = ['(words + list)', '(words)']
    types_qa = []
    
    # get uniques
    level_parse = 'Level 1'
    plot_level = 'Plot-level questions'
    qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types_qa, use_split_keys=False)
    
    level_parse = 'Level 2'
    plot_level = 'Plot-level questions'
    qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types_qa, use_split_keys=False)
    
    level_parse = 'Level 3'
    plot_level = 'Plot-level questions'
    qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types_qa, use_split_keys=False)

    responses = []; prompts = []; system_prompts = []
    for question_list in qa:
        response, prompt, system_prompt_out = send_to_gemini(question_list, img_path, client,
                    test_run = test_run, 
                    verbose=verbose, img_format=img_format,
                    system_prompt = system_prompt, config=config, reasoning = reasoning_text,
                    model_name=model_name, thinking_level=thinking_level)
        responses.append(response)
        question_list['prompt'] = prompt
        question_list['system prompt'] = system_prompt_out
        #import sys; sys.exit()


    # parse for errors
    print('')
    print('**** Cleaned QA ****')
    qa = parse_for_errors(qa, llm='Gemini')
    #qa = parse_for_errors(qa) # might need to do this again

    # dump to file
    if not test_run:
        with open(dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle', 'wb') as ff:
            pickle.dump([qa, model_name], ff)
        print('Just saved:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
    else:
        print('Would store at:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
    #import sys; sys.exit()

print('!!!!!!!! DONE !!!!!!!!!!')


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


on 0 of 255
have file already: /Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150/gemini/vqa_000007_qa.pickle
on 1 of 255
have file already: /Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150/gemini/vqa_000010_qa.pickle
on 2 of 255
have file already: /Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150/gemini/vqa_000011_qa.pickle
on 3 of 255
have file already: /Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150/gemini/vqa_000013_qa.pickle
on 4 of 255
have file already: /Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150/gemini/vqa_000017_qa.pickle
on 5 of 255
have file already: /Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150/gemini/vqa_000023_qa.pickle
on 6 of 255
have file already: /Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150/gemini/vqa_000024_qa.pickle
on 7 of 255
have file already: /Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150/gemini/vqa_000025_qa.pickle
on 8 of 255
have file already: /Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150/gemini/vqa_000030_qa.pickle
o

     response: ```json
{"plot style": "grayscale"}
```

```json
{"explanation": "The figure exhibits a distinct gray background and specific font styling and color schemes that are characteristic of the 'grayscale' style in matplotlib, or alternatively a customized dark/gray theme commonly mimicking specific publication presets using grayscale properties."}
```
   on question: assume this is a figure made with matplotlib in Python. Examples of matplotlib colormaps are "rainbow" or "Reds". What is the colormap that was used in this figure? Please format the output as a json as {"colormap":""} to store the matplotlib colormap used in the figure. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.


     response: {
  "colormap": "RdBu_r"
}

{
  "explanation": "The colorbar transitions from blue on the left to white in the middle, and to red on the right. This is characteristic of the divergent 'RdBu' colormap, and since lower values (left) correspond to blue and higher values (right) to red, the reversed version 'RdBu_r' was used."
}
   on question: what is the aspect ratio of this figure? Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.


     response: ```json
{"aspect ratio": "3:4"}
```

```json
{"explanation": "The image has a vertical orientation where the height is noticeably greater than the width. By visually comparing the dimensions of the main plot area and the overall image bounding box, the width spans approximately 3 units on the horizontal axis (from 0 to 4 with ticks at 0, 2, 4) and the height spans approximately 5 units on the vertical axis (from 0 to 5), giving a rough data-space aspect ratio of 3:5, but the overall figure bounding box closely approximates a standard portrait ratio of approximately 3:4."}
```
   on question: what is the title of the plot in this figure? Please format the output as a json as {"titles":[]}, where the list holds the title of the plot as its only element.  If the plot does not have a title, then denote this by an empty string in the list.  Please format any formulas in the title in a Python LaTeX string (for example 'Light $\\alpha$').  In addition to providing your answer, 

...

     response: ```json
{
  "mean color": "15900.0"
}
```
```json
{
  "explanation": "The colorbar represents a linear scale ranging from 15600 to 16200, with tick marks at 15600, 15800, 16000, and 16200. Assuming a uniform or normally distributed dataset across this range (typical for synthetic noise grids like this), the mean value corresponds to the midpoint of the color scale. The midpoint between 15600 and 16200 is (15600 + 16200) / 2 = 15900.0."
}
```
   on question: what is the median value of the data along the color-axis in this figure?  Please format the output as a json as {"median color":""} for this figure, where the "median color" value should be a float, calculated from the data values used to create the plot. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, 

     response: ```json
{"median color": "15900"}
```

```json
{"explanation": "The colorbar at the top of the figure represents values ranging uniformly from 15600 (magenta/red start) to 16200 (red end). The median value of this continuous linear scale is simply the midpoint between the minimum and maximum values, which is (15600 + 16200) / 2 = 15900."}
```
   on question: what is the minimum value of the data along the color-axis in this figure?  Please format the output as a json as {"minimum color":""} for this figure, where the "minimum color" value should be a float, calculated from the data values used to create the plot. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.


     response: ```json
{
  "minimum color": "15500.0"
}
```
```json
{
  "explanation": "Looking at the color bar at the top of the image, the numerical values corresponding to the ticks are 15600, 15800, 16000, and 16200. The color bar starts slightly to the left of the '15600' label, with the magenta/purple end representing the minimum value. Given the uniform spacing of 200 units between the labeled major ticks (15600 to 15800), and the extent of the color bar past the 15600 tick to the left edge of the color scale by about half a division, the minimum value represented at the far left edge of the color-axis is 15500.0."
}
```
   on question: please choose the distribution from the following list: [gaussian mixture model, real image of the sky]. What is the underlying distribution used to create the data in this figure? Please format the output as a json as {"distribution":""} for this figure, where the "distribution" value should be a string. In addition to providing your answer, pl

     response: {
  "distribution": "real image of the sky"
}
{
  "explanation": "The image displays astronomical coordinates with Right Ascension and Declination axes along with a color bar scaled with values typical of astronomical data, titled 'UNIVERSITY SKY CLUSTERS CODES', indicating it represents a real image or map of the sky rather than a synthetic Gaussian mixture model."
}
   on question: approximately what angular size does a single pixel of this sky image span? Give the value in arcseconds. Please format the output as a json as {"pixel scale":""} for this figure, where the value should be a float, expressed in arcseconds per pixel. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.


     response: ```json
{
  "pixel scale": "3.375"
}
```

```json
{
  "explanation": "To find the pixel scale, we first determine the total angular span in Declination and Right Ascension covered by the image. The Declination goes from -26° 15' at the top to -27° 15' at the bottom, which is a total span of 1 degree, or 60 arcminutes (3600 arcseconds). The Right Ascension spans from approximately 2h 12m at the left to 2h 08m at the right, which is 4 minutes of time. At a declination of roughly -26.5°, 4 minutes of RA corresponds to an angular width of 4 * 15 arcminutes * cos(-26.5°) = 60 * 0.895 = 53.7 arcminutes (3222 arcseconds). By counting the number of pixels across the width and height of the grid (approximately 950-1000 pixels vertically for 3600 arcseconds), we divide the total angular span by the number of pixels to arrive at a pixel scale of approximately 3.375 arcseconds per pixel."
}
```

**** Cleaned QA ****
Q: I am going to show you an image. Now, assume this is a figure ma

In [9]:
#question

## Look at data

Check out one, if you wanna:

In [10]:
pickles = glob(dir_api + '*.pickle')
#pickles = glob('/Users/jnaiman/Downloads/tmp/JCDL2025/example_hists/claude_api/*pickle')
pickles[:5]

['/Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150/gemini/vqa_001556_qa.pickle',
 '/Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150/gemini/vqa_000146_qa.pickle',
 '/Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150/gemini/vqa_000977_qa.pickle',
 '/Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150/gemini/vqa_000014_qa.pickle',
 '/Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150/gemini/vqa_000128_qa.pickle']

In [11]:
ifile = 0
with open(pickles[ifile], 'rb') as f:
    qa_in = pickle.load(f)[0]

In [12]:
qa_in[0]

{'Q': 'You are a helpful assistant that can analyze images. Assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot". What is the plot style used in this figure? Please format the output as a json as {"plot style":""} to store the matplotlib plotting style used in the figure.',
 'A': {'plot style': 'grayscale'},
 'Level': 'Level 1',
 'type': 'Figure-level questions',
 'Response': {'explanation': "The figure exhibits characteristic visual features of the matplotlib 'classic' style, including a solid grey background, black axes borders with inward-facing ticks, serif-style fonts, and a distinct layout of the colorbar and tick labels."},
 'persona': 'You are a helpful assistant that can analyze images.',
 'context': 'Assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot".',
 'question': 'What is the plot style used in this figure?',
 'format': 'Please format the output as a json a

In [13]:
#qa_in = parse_for_errors_claude(qa_in, llm='gemini')

Claude outputs reasoning, so we have to do a bit of cleaning from the responses:

In [14]:
print(pickles[ifile])
print('*********')
for qa_pairs in qa_in:
    print('Prompt:', qa_pairs['prompt'])
    print('  Real A:', qa_pairs['A'])
    print('Gemini A:', qa_pairs['raw answer'])
    print('    input tokens:', qa_pairs['usage']['input_tokens'])
    print('    output tokens:', qa_pairs['usage']['output_tokens'])
    print('')

/Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150/gemini/vqa_001556_qa.pickle
*********
Prompt: I am going to show you an image. Now, assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot". What is the plot style used in this figure? Please format the output as a json as {"plot style":""} to store the matplotlib plotting style used in the figure. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
  Real A: {'plot style': 'grayscale'}
Gemini A: ```json
{"plot style": "classic"}
```
```json
{"explanation": "The figure exhibits characteristic visual features of the matplotlib 'classic' style, including a solid grey background, black axes borders with inward-facing ticks, serif-style fonts, and a distinct l